In [ ]:
"""
============================================================
  TDNN (Time Delay Neural Network) — Javanese ASR
  Perbandingan dengan GMM pipeline
============================================================
  Input  : File .wav dari SAMPLED_ROOT
           + dataSampledSmall.csv (kolom Path_Data + sentences)
  Proses : Ekstrak MFCC 39-dim per frame dari .wav (librosa)
           Label per frame = kata-kata dari sentences (audio-level)
           Training TDNN frame-level → Dekode → WER + Akurasi
  Output : HasilWER_TDNN.csv
           HasilDekode_TDNN.csv
           HasilAkurasi_TDNN.csv
           HasilTraining_TDNN.csv
  Split  : 80% train / 10% val / 10% test (dari 1749 audio)
============================================================
"""

import os
import time
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
#  KONFIGURASI — SESUAIKAN DI SINI
# ─────────────────────────────────────────────────────────────

# ── Path root folder audio (WAV/FLAC) ──
SAMPLED_ROOT = r"D:\doc\2026\FINAL_CODE\data\sampled_small"

# ── Path CSV transkrip hasil stratified sampling ──
CSV_PATH = (
    r"D:/doc/2026/FINAL_CODE/src/codeStat/preModelling/"
    r"samplingStratified/out_samplingStratified/dataSampledSmall.csv"
)

# ── Folder output hasil ──
DIR_SAVE = (
    r"D:/doc/2026/FINAL_CODE/src/codeStat/modelling/"
    r"voiceModelling/srcVoiceModelling/GMM_TDNN/out/TDNN"
)

os.makedirs(DIR_SAVE, exist_ok=True)

# ── Path output ──
OUT_WER_CSV     = os.path.join(DIR_SAVE, "HasilWER_TDNN.csv")
OUT_DEC_CSV     = os.path.join(DIR_SAVE, "HasilDekode_TDNN.csv")
OUT_ACC_CSV     = os.path.join(DIR_SAVE, "HasilAkurasi_TDNN.csv")
OUT_HISTORY_CSV = os.path.join(DIR_SAVE, "HasilTraining_TDNN.csv")
MODEL_PATH      = os.path.join(DIR_SAVE, "tdnn_best.npy")

# ── Kolom dataSampledSmall.csv ──
COL_PATH_DATA = "Path_Data"    # kolom path file WAV
COL_SENTENCES = "sentences"    # kolom kalimat transkripsi

# ── Parameter ekstraksi MFCC (sesuai kode GMM) ──
SR          = 16000
N_MFCC      = 13
N_FFT       = 512
HOP_LENGTH  = 160
WIN_LENGTH  = 400
N_MELS      = 26
PRE_EMPH    = 0.97
FMIN        = 0.0
FMAX        = 4000.0
DELTA_WIDTH = 2
INPUT_DIM   = N_MFCC * 3      # 13 MFCC + 13 delta + 13 delta2 = 39

# ── Arsitektur TDNN ──
TDNN_CONTEXTS = [
    [-2,  2],   # Layer 1: ±2 frame  → konteks lokal
    [-2,  0],   # Layer 2: 2 masa lalu → temporal history
    [-3,  3],   # Layer 3: ±3 frame  → konteks lebar
    [ 0,  0],   # Layer 4: frame saat ini → bottleneck
]
HIDDEN_DIMS   = [512, 512, 512, 512]
DROPOUT_RATE  = 0.1
LEARNING_RATE = 0.001

# ── Pelatihan ──
BATCH_SIZE  = 256
MAX_EPOCHS  = 30
EARLY_STOP  = 5

# ── Split dataset ──
TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
RANDOM_SEED = 42


# ─────────────────────────────────────────────────────────────
#  EKSTRAKSI MFCC DARI WAV
#  (kode dari pipeline GMM — tidak diubah)
# ─────────────────────────────────────────────────────────────

try:
    import librosa
    LIBROSA_AVAILABLE = True
except ImportError:
    LIBROSA_AVAILABLE = False

try:
    from scipy.io import wavfile
    SCIPY_AVAILABLE = True
except ImportError:
    SCIPY_AVAILABLE = False


def extract_features(wav_path: str) -> np.ndarray:
    if not os.path.isfile(wav_path):
        raise FileNotFoundError(f"File tidak ditemukan: {wav_path}")
    if LIBROSA_AVAILABLE:
        return _extract_librosa(wav_path)
    else:
        return _extract_manual(wav_path)


def _extract_librosa(wav_path: str) -> np.ndarray:
    y, sr = librosa.load(wav_path, sr=SR, mono=True)
    if len(y) == 0:
        raise ValueError(f"Audio kosong: {wav_path}")

    # Pre-emphasis
    y_emph = np.append(y[0], y[1:] - PRE_EMPH * y[:-1])

    # MFCC 13 koefisien
    mfcc = librosa.feature.mfcc(
        y=y_emph, sr=SR, n_mfcc=N_MFCC,
        n_fft=N_FFT, hop_length=HOP_LENGTH,
        win_length=WIN_LENGTH, n_mels=N_MELS,
        fmin=FMIN, fmax=FMAX, window="hamming",
    )  # (13, T)

    delta  = librosa.feature.delta(mfcc, width=DELTA_WIDTH * 2 + 1)
    delta2 = librosa.feature.delta(mfcc, order=2, width=DELTA_WIDTH * 2 + 1)

    return np.vstack([mfcc, delta, delta2]).T.astype(np.float32)  # (T, 39)


def _extract_manual(wav_path: str) -> np.ndarray:
    if SCIPY_AVAILABLE:
        sr, audio = wavfile.read(wav_path)
        if audio.dtype == np.int16:
            audio = audio.astype(np.float32) / 32768.0
        elif audio.dtype == np.int32:
            audio = audio.astype(np.float32) / 2147483648.0
    else:
        raise ImportError("librosa atau scipy diperlukan")

    if sr != SR:
        n_new = int(len(audio) * SR / sr)
        audio = np.interp(
            np.linspace(0, len(audio) - 1, n_new),
            np.arange(len(audio)), audio
        )

    y_emph = np.append(audio[0], audio[1:] - PRE_EMPH * audio[:-1])
    frames = _framing(y_emph, WIN_LENGTH, HOP_LENGTH)
    frames = frames * np.hamming(WIN_LENGTH)

    fft_mag    = np.abs(np.fft.rfft(frames, n=N_FFT))
    mel_fb     = _mel_filterbank(SR, N_FFT, N_MELS, FMIN, FMAX)
    mel_energy = np.dot(fft_mag ** 2, mel_fb.T)
    mel_energy = np.where(mel_energy > 0, mel_energy, 1e-10)
    log_mel    = np.log10(mel_energy)
    mfcc       = _dct(log_mel, N_MFCC)
    delta      = _compute_delta(mfcc, DELTA_WIDTH)
    delta2     = _compute_delta(delta, DELTA_WIDTH)

    return np.concatenate([mfcc, delta, delta2], axis=1).astype(np.float32)


def _framing(y, win_len, hop_len):
    n_frames = 1 + (len(y) - win_len) // hop_len
    frames   = np.zeros((n_frames, win_len), dtype=np.float32)
    for i in range(n_frames):
        frames[i] = y[i * hop_len: i * hop_len + win_len]
    return frames


def _mel_filterbank(sr, n_fft, n_mels, fmin, fmax):
    def hz_to_mel(f): return 1127.0 * np.log1p(f / 700.0)
    def mel_to_hz(m): return 700.0 * (np.exp(m / 1127.0) - 1.0)

    mel_pts  = np.linspace(hz_to_mel(fmin), hz_to_mel(fmax), n_mels + 2)
    hz_pts   = mel_to_hz(mel_pts)
    bins     = np.floor((n_fft + 1) * hz_pts / sr).astype(int)
    fb       = np.zeros((n_mels, n_fft // 2 + 1))
    for m in range(1, n_mels + 1):
        for k in range(bins[m-1], bins[m]):
            fb[m-1, k] = (k - bins[m-1]) / (bins[m] - bins[m-1] + 1e-10)
        for k in range(bins[m], bins[m+1]):
            fb[m-1, k] = (bins[m+1] - k) / (bins[m+1] - bins[m] + 1e-10)
    return fb


def _dct(x, n_coeff):
    T, N  = x.shape
    n_idx = np.arange(n_coeff)
    k_idx = np.arange(N) + 0.5
    dct_m = np.cos(np.pi * n_idx[:, None] * k_idx[None, :] / N)
    return (x @ dct_m.T) * np.sqrt(2.0 / N)


def _compute_delta(feat, width=2):
    T, D   = feat.shape
    delta  = np.zeros_like(feat)
    denom  = 2.0 * sum(i**2 for i in range(1, width + 1))
    for t in range(T):
        for n in range(1, width + 1):
            delta[t] += n * (feat[min(t+n, T-1)] - feat[max(t-n, 0)])
    return delta / denom


# ─────────────────────────────────────────────────────────────
#  DATA LOADER
#  Baca CSV → temukan .wav → ekstrak MFCC per frame
#  Label per frame = kalimat audio-level (dari sentences)
# ─────────────────────────────────────────────────────────────

def find_wav(audio_id: str) -> str:
    """
    Cari file .wav di dalam SAMPLED_ROOT secara rekursif.
    Mendukung struktur folder flat maupun nested.
    """
    # Coba langsung dulu (flat)
    direct = os.path.join(SAMPLED_ROOT, audio_id + ".wav")
    if os.path.isfile(direct):
        return direct

    # Rekursif ke subfolder
    for root, _, files in os.walk(SAMPLED_ROOT):
        for f in files:
            if os.path.splitext(f)[0] == audio_id and f.endswith(".wav"):
                return os.path.join(root, f)
    return ""


def load_data_from_wav(csv_path: str) -> tuple:
    """
    Baca dataSampledSmall.csv → untuk setiap audio:
      1. Temukan file .wav via find_wav()
      2. Ekstrak MFCC per frame: (T, 39)
      3. Label per frame = indeks kata dalam sentences (word-level)
         atau indeks audio (audio-level jika kalimat tidak dipecah)

    Strategi label:
      - Setiap audio punya kalimat (misal: "aku arep menyang pasar")
      - Kalimat dipecah jadi kata-kata → vocab global
      - Label per frame = indeks kalimat (audio-level label)
        → satu label untuk seluruh frame audio tersebut
        → cocok untuk klasifikasi ujaran (utterance classification)

    Return:
      sequences  : {audio_id: np.array (T, 39)}
      labels     : {audio_id: np.array (T,) int}   ← label sama untuk semua frame
      label2id   : {kalimat_str: int}
      id2label   : {int: kalimat_str}
      word_refs  : {audio_id: [kata1, kata2, ...]} ← untuk WER
    """
    print(f"  Membaca {csv_path} ...")
    df = pd.read_csv(csv_path)
    print(f"  Shape CSV : {df.shape}")
    print(f"  Kolom     : {list(df.columns)}")

    # Validasi kolom
    for col in [COL_PATH_DATA, COL_SENTENCES]:
        if col not in df.columns:
            raise ValueError(f"Kolom '{col}' tidak ditemukan. Kolom ada: {list(df.columns)}")

    # Bangun label: kalimat → int (utterance-level classification)
    # Setiap kalimat unik jadi satu kelas
    sentences_list = df[COL_SENTENCES].astype(str).str.lower().str.strip().tolist()
    unique_sents   = sorted(set(sentences_list))
    label2id       = {s: i for i, s in enumerate(unique_sents)}
    id2label       = {i: s for s, i in label2id.items()}
    print(f"  Kalimat unik (kelas): {len(label2id)}")

    sequences = {}
    labels    = {}
    word_refs = {}
    skipped   = 0

    print(f"  Mengekstrak MFCC dari {len(df)} file .wav ...")
    for idx, row in df.iterrows():
        audio_id = os.path.splitext(os.path.basename(str(row[COL_PATH_DATA])))[0]
        kalimat  = str(row[COL_SENTENCES]).lower().strip()
        label_id = label2id[kalimat]
        words    = kalimat.split()

        # Cari .wav
        wav_path = str(row[COL_PATH_DATA])
        if not os.path.isfile(wav_path):
            wav_path = find_wav(audio_id)

        if not wav_path:
            print(f"  ⚠ Tidak ditemukan: {audio_id} — dilewati")
            skipped += 1
            continue

        try:
            feats = extract_features(wav_path)   # (T, 39)
            T     = feats.shape[0]
            if T == 0:
                skipped += 1
                continue

            sequences[audio_id] = feats
            # Label per frame = label kalimat (sama untuk semua frame)
            labels[audio_id]    = np.full(T, label_id, dtype=np.int32)
            word_refs[audio_id] = words

        except Exception as e:
            print(f"  ⚠ Error {audio_id}: {e} — dilewati")
            skipped += 1
            continue

        if (idx + 1) % 100 == 0 or (idx + 1) == len(df):
            print(f"    [{idx+1}/{len(df)}] diproses, {skipped} dilewati")

    total_frames = sum(v.shape[0] for v in sequences.values())
    actual_dim   = next(iter(sequences.values())).shape[1] if sequences else INPUT_DIM

    print(f"  Audio berhasil : {len(sequences)}")
    print(f"  Audio dilewati : {skipped}")
    print(f"  Total frame    : {total_frames:,}")
    print(f"  Input dim      : {actual_dim}")

    return sequences, labels, label2id, id2label, word_refs, actual_dim


# ─────────────────────────────────────────────────────────────
#  AKTIVASI & LOSS (NumPy only)
# ─────────────────────────────────────────────────────────────

def relu(x):
    return np.maximum(0.0, x)

def relu_grad(x):
    return (x > 0).astype(np.float32)

def softmax(x):
    x = x - np.max(x, axis=-1, keepdims=True)
    e = np.exp(x)
    return e / (e.sum(axis=-1, keepdims=True) + 1e-9)

def cross_entropy_loss(probs, labels):
    n  = len(labels)
    lp = np.log(probs[np.arange(n), labels] + 1e-9)
    return -lp.mean()

def frame_accuracy(probs, labels):
    return (np.argmax(probs, axis=1) == labels).mean()


# ─────────────────────────────────────────────────────────────
#  TDNN LAYER
# ─────────────────────────────────────────────────────────────

class TDNNLayer:
    def __init__(self, in_dim: int, out_dim: int, context: list, lr: float = 0.001):
        self.context = context
        self.ctx_len = context[1] - context[0] + 1
        self.lr      = lr

        fan_in = in_dim * self.ctx_len
        scale  = np.sqrt(2.0 / fan_in)
        self.W = (np.random.randn(fan_in, out_dim) * scale).astype(np.float32)
        self.b = np.zeros(out_dim, dtype=np.float32)

        # Adam state
        self.mW = np.zeros_like(self.W)
        self.vW = np.zeros_like(self.W)
        self.mb = np.zeros_like(self.b)
        self.vb = np.zeros_like(self.b)
        self.t  = 0
        self._cache = None

    def splice(self, X: np.ndarray) -> np.ndarray:
        """
        Context splicing dengan index-clamp — aman untuk T=1 sekalipun.
        X: (T, D) → return (T, D * ctx_len)
        """
        T        = X.shape[0]
        base_idx = np.arange(T)
        parts    = []
        for offset in range(self.context[0], self.context[1] + 1):
            shifted = np.clip(base_idx + offset, 0, T - 1)
            parts.append(X[shifted])
        return np.concatenate(parts, axis=1)

    def forward(self, X_spliced: np.ndarray) -> np.ndarray:
        self._cache = X_spliced
        return X_spliced @ self.W + self.b

    def backward(self, grad_out: np.ndarray) -> np.ndarray:
        batch = grad_out.shape[0]
        dW    = self._cache.T @ grad_out / batch
        db    = grad_out.mean(axis=0)
        dX    = grad_out @ self.W.T
        self._adam(dW, db)
        return dX

    def _adam(self, dW, db, b1=0.9, b2=0.999, eps=1e-8):
        self.t  += 1
        self.mW  = b1 * self.mW + (1 - b1) * dW
        self.vW  = b2 * self.vW + (1 - b2) * dW**2
        self.W  -= self.lr * (self.mW / (1 - b1**self.t)) / (np.sqrt(self.vW / (1 - b2**self.t)) + eps)
        self.mb  = b1 * self.mb + (1 - b1) * db
        self.vb  = b2 * self.vb + (1 - b2) * db**2
        self.b  -= self.lr * (self.mb / (1 - b1**self.t)) / (np.sqrt(self.vb / (1 - b2**self.t)) + eps)


# ─────────────────────────────────────────────────────────────
#  MODEL TDNN
# ─────────────────────────────────────────────────────────────

class TDNNModel:
    def __init__(self, input_dim, hidden_dims, contexts, output_dim,
                 dropout=0.1, lr=0.001):
        assert len(hidden_dims) == len(contexts)
        self.dropout  = dropout
        self.training = True

        self.layers = []
        in_d = input_dim
        for h_dim, ctx in zip(hidden_dims, contexts):
            self.layers.append(TDNNLayer(in_d, h_dim, ctx, lr))
            in_d = h_dim
        self.layers.append(TDNNLayer(in_d, output_dim, [0, 0], lr))

        self._relu_cache = []
        self._drop_cache = []

    def forward(self, X: np.ndarray) -> np.ndarray:
        """X: (T, input_dim) → logits: (T, output_dim)"""
        self._relu_cache = []
        self._drop_cache = []
        h = X

        for layer in self.layers[:-1]:
            z = layer.forward(layer.splice(h))
            a = relu(z)
            if self.training and self.dropout > 0:
                mask  = (np.random.rand(*a.shape) > self.dropout).astype(np.float32)
                a     = a * mask / (1.0 - self.dropout)
            else:
                mask = np.ones(a.shape, dtype=np.float32)
            self._relu_cache.append(z)
            self._drop_cache.append(mask)
            h = a

        out = self.layers[-1]
        return out.forward(out.splice(h))

    def loss_and_backward(self, X, labels):
        logits = self.forward(X)
        probs  = softmax(logits)
        loss   = cross_entropy_loss(probs, labels)
        acc    = frame_accuracy(probs, labels)

        T    = len(labels)
        dlog = probs.copy()
        dlog[np.arange(T), labels] -= 1.0

        dh = self.layers[-1].backward(dlog)
        scale = 1.0 / (1.0 - self.dropout + 1e-9) if self.dropout > 0 else 1.0
        for i in reversed(range(len(self.layers) - 1)):
            dh = dh * self._drop_cache[i] * scale
            dh = dh * relu_grad(self._relu_cache[i])
            dh = self.layers[i].backward(dh)

        return loss, acc

    def predict_proba(self, X):
        self.training = False
        return softmax(self.forward(X))

    def save(self, path):
        np.save(path, {"weights": [(l.W.copy(), l.b.copy()) for l in self.layers]},
                allow_pickle=True)

    def load(self, path):
        data = np.load(path, allow_pickle=True).item()
        for i, (W, b) in enumerate(data["weights"]):
            self.layers[i].W = W
            self.layers[i].b = b
        print(f"  Bobot terbaik dimuat dari {path}")


# ─────────────────────────────────────────────────────────────
#  DATASET SPLIT  80 / 10 / 10
# ─────────────────────────────────────────────────────────────

def split_dataset(audio_ids):
    rng  = np.random.default_rng(RANDOM_SEED)
    ids  = list(audio_ids)
    perm = rng.permutation(len(ids))
    n    = len(ids)
    n_tr = int(n * TRAIN_RATIO)
    n_va = int(n * VAL_RATIO)

    train = [ids[i] for i in perm[:n_tr]]
    val   = [ids[i] for i in perm[n_tr:n_tr + n_va]]
    test  = [ids[i] for i in perm[n_tr + n_va:]]

    print(f"  Split  →  train: {len(train)}  val: {len(val)}  test: {len(test)}")
    return train, val, test


# ─────────────────────────────────────────────────────────────
#  PELATIHAN
# ─────────────────────────────────────────────────────────────

def train(model, train_ids, val_ids, sequences, labels):
    rng     = np.random.default_rng(RANDOM_SEED)
    history = []
    best_val_loss  = float("inf")
    patience_count = 0

    print(f"\n  Training: {len(train_ids)} audio  |  Val: {len(val_ids)} audio")
    print(f"  Epoch max: {MAX_EPOCHS}  |  Early stop: {EARLY_STOP}  |  Batch: {BATCH_SIZE}")
    print(f"\n  {'Epoch':>5}  {'TrainLoss':>10}  {'TrainAcc':>9}  "
          f"{'ValLoss':>8}  {'ValAcc':>8}  {'Waktu':>7}")
    print(f"  {'-'*58}")

    for epoch in range(1, MAX_EPOCHS + 1):
        t0 = time.time()
        model.training = True
        rng.shuffle(train_ids)

        ep_loss = ep_acc = 0.0
        n_batch = 0

        for audio_id in train_ids:
            X = sequences[audio_id]   # (T, 39)
            y = labels[audio_id]      # (T,)
            T = len(X)

            idx = rng.permutation(T)
            for start in range(0, T, BATCH_SIZE):
                sl = idx[start:start + BATCH_SIZE]
                if len(sl) < 2:
                    continue
                loss, acc = model.loss_and_backward(X[sl], y[sl])
                ep_loss  += loss
                ep_acc   += acc
                n_batch  += 1

        ep_loss /= max(n_batch, 1)
        ep_acc  /= max(n_batch, 1)

        # Validasi
        model.training = False
        vl = va = 0.0
        for audio_id in val_ids:
            X = sequences[audio_id]
            y = labels[audio_id]
            p = model.predict_proba(X)
            vl += cross_entropy_loss(p, y)
            va += frame_accuracy(p, y)
        vl /= max(len(val_ids), 1)
        va /= max(len(val_ids), 1)

        elapsed = time.time() - t0
        history.append({
            "epoch": epoch, "train_loss": round(ep_loss, 6),
            "train_acc": round(ep_acc, 6),
            "val_loss": round(vl, 6), "val_acc": round(va, 6),
        })
        print(f"  {epoch:5d}  {ep_loss:10.4f}  {ep_acc:9.4f}  "
              f"{vl:8.4f}  {va:8.4f}  {elapsed:6.1f}s")

        if vl < best_val_loss - 1e-5:
            best_val_loss  = vl
            patience_count = 0
            model.save(MODEL_PATH)
        else:
            patience_count += 1
            if patience_count >= EARLY_STOP:
                print(f"\n  ✓ Early stopping pada epoch {epoch} "
                      f"(best val_loss={best_val_loss:.4f})")
                break

    return history


# ─────────────────────────────────────────────────────────────
#  WER
# ─────────────────────────────────────────────────────────────

def levenshtein(ref, hyp):
    R, H = len(ref), len(hyp)
    dp   = np.zeros((R+1, H+1), dtype=int)
    dp[:,0] = np.arange(R+1)
    dp[0,:] = np.arange(H+1)
    for i in range(1, R+1):
        for j in range(1, H+1):
            cost    = 0 if ref[i-1] == hyp[j-1] else 1
            dp[i,j] = min(dp[i-1,j]+1, dp[i,j-1]+1, dp[i-1,j-1]+cost)

    i, j = R, H
    S = D = I = 0
    while i > 0 or j > 0:
        if i > 0 and j > 0:
            cost = 0 if ref[i-1] == hyp[j-1] else 1
            if dp[i,j] == dp[i-1,j-1] + cost:
                if cost: S += 1
                i -= 1; j -= 1
            elif dp[i,j] == dp[i-1,j] + 1:
                D += 1; i -= 1
            else:
                I += 1; j -= 1
        elif i > 0:
            D += 1; i -= 1
        else:
            I += 1; j -= 1

    return min((S+D+I)/max(R,1), 1.0), S, D, I, max(R, 1)


def decode_sequence(probs, id2label):
    """
    Dekode frame-level probs → kalimat.
    Ambil label mayoritas (voting) sebagai prediksi kalimat,
    lalu pecah kalimat tersebut jadi kata-kata.
    """
    pred_frames = np.argmax(probs, axis=1)   # (T,)
    # Majority vote: label yang paling sering muncul
    unique, counts = np.unique(pred_frames, return_counts=True)
    best_label_id  = unique[np.argmax(counts)]
    kalimat_pred   = id2label.get(int(best_label_id), "")
    return kalimat_pred.split()


# ─────────────────────────────────────────────────────────────
#  EVALUASI: WER + AKURASI
# ─────────────────────────────────────────────────────────────

def evaluate(model, eval_ids, sequences, labels, id2label, word_refs):
    model.training = False
    print(f"\n  Mendekode {len(eval_ids)} audio ...")

    wer_rows, acc_rows = [], []
    total_S = total_D = total_I = total_N = 0
    total_correct = total_frames = 0

    for audio_id in eval_ids:
        X      = sequences[audio_id]
        y_true = labels[audio_id]
        T      = len(X)

        probs       = model.predict_proba(X)
        pred_frames = np.argmax(probs, axis=1)

        # Akurasi frame
        correct       = int((pred_frames == y_true).sum())
        total_correct += correct
        total_frames  += T

        # Dekode → kata
        hyp_words = decode_sequence(probs, id2label)
        ref_words = word_refs.get(audio_id, [])

        wer_val, S, D, I, N = levenshtein(ref_words, hyp_words)
        total_S += S; total_D += D; total_I += I; total_N += N

        wer_rows.append({
            "audio_id" : audio_id,
            "WER"      : round(wer_val * 100, 4),
            "S": S, "D": D, "I": I, "N": N,
            "referensi": " ".join(ref_words),
            "hipotesis": " ".join(hyp_words),
        })
        acc_rows.append({
            "audio_id"   : audio_id,
            "akurasi_pct": round(correct / T * 100, 4),
            "benar"      : correct,
            "total_frame": T,
        })

    wer_corpus = (total_S + total_D + total_I) / max(total_N, 1) * 100
    acc_global = total_correct / max(total_frames, 1) * 100
    wer_list   = [r["WER"] for r in wer_rows]
    acc_list   = [r["akurasi_pct"] for r in acc_rows]

    summary = {
        "n_audio"    : len(eval_ids),
        "wer_corpus" : round(wer_corpus, 4),
        "wer_mean"   : round(float(np.mean(wer_list)), 4),
        "wer_min"    : round(float(np.min(wer_list)), 4),
        "wer_max"    : round(float(np.max(wer_list)), 4),
        "acc_global" : round(acc_global, 4),
        "acc_mean"   : round(float(np.mean(acc_list)), 4),
        "acc_min"    : round(float(np.min(acc_list)), 4),
        "acc_max"    : round(float(np.max(acc_list)), 4),
        "total_S"    : total_S, "total_D": total_D,
        "total_I"    : total_I, "total_N": total_N,
        "total_frame": total_frames, "total_benar": total_correct,
    }

    _print_stats(summary)
    return wer_rows, acc_rows, summary


def _print_stats(s):
    print(f"\n── Statistik Evaluasi TDNN ────────────────────────────────")
    print(f"  Jumlah audio     : {s['n_audio']}")
    print(f"  ── WER ──────────────────────────────────────────────────")
    print(f"  WER Corpus       : {s['wer_corpus']:.2f}%")
    print(f"  WER Mean         : {s['wer_mean']:.2f}%")
    print(f"  WER Min          : {s['wer_min']:.2f}%")
    print(f"  WER Max          : {s['wer_max']:.2f}%")
    print(f"  Total S          : {s['total_S']}")
    print(f"  Total D          : {s['total_D']}")
    print(f"  Total I          : {s['total_I']}")
    print(f"  Total N (kata)   : {s['total_N']}")
    print(f"  ── Akurasi Frame ────────────────────────────────────────")
    print(f"  Akurasi Global   : {s['acc_global']:.2f}%")
    print(f"  Akurasi Mean     : {s['acc_mean']:.2f}%")
    print(f"  Akurasi Min      : {s['acc_min']:.2f}%")
    print(f"  Akurasi Max      : {s['acc_max']:.2f}%")
    print(f"  Total frame      : {s['total_frame']:,}")
    print(f"  Frame benar      : {s['total_benar']:,}")
    print(f"────────────────────────────────────────────────────────────")


# ─────────────────────────────────────────────────────────────
#  PERBANDINGAN GMM vs TDNN
# ─────────────────────────────────────────────────────────────

def print_comparison(summary):
    gmm = {
        "wer_corpus": 99.84, "wer_mean": 99.82,
        "wer_min": 50.00,    "wer_max": 100.00,
        "total_S": 1730,     "total_D": 10394,
        "total_I": 0,        "total_N": 12143,
    }
    t = summary

    print(f"\n{'='*62}")
    print(f"  PERBANDINGAN GMM vs TDNN")
    print(f"{'='*62}")
    print(f"  {'Metrik':<30} {'GMM':>10} {'TDNN':>10}")
    print(f"  {'-'*52}")
    for label, g, td in [
        ("WER Corpus (%)",       f"{gmm['wer_corpus']:.2f}", f"{t['wer_corpus']:.2f}"),
        ("WER Mean (%)",         f"{gmm['wer_mean']:.2f}",   f"{t['wer_mean']:.2f}"),
        ("WER Min (%)",          f"{gmm['wer_min']:.2f}",    f"{t['wer_min']:.2f}"),
        ("WER Max (%)",          f"{gmm['wer_max']:.2f}",    f"{t['wer_max']:.2f}"),
        ("Total S (substitusi)", str(gmm['total_S']),        str(t['total_S'])),
        ("Total D (deletion)",   str(gmm['total_D']),        str(t['total_D'])),
        ("Total I (insertion)",  str(gmm['total_I']),        str(t['total_I'])),
        ("Total N (kata ref)",   str(gmm['total_N']),        str(t['total_N'])),
        ("Akurasi Frame (%)",    "N/A",                      f"{t['acc_global']:.2f}"),
    ]:
        print(f"  {label:<30} {g:>10} {td:>10}")
    print(f"{'='*62}")

    delta = gmm['wer_corpus'] - t['wer_corpus']
    if delta > 0:
        print(f"  ✓ TDNN lebih baik dari GMM sebesar {delta:.2f}% WER")
    elif delta < 0:
        print(f"  ✗ TDNN lebih buruk dari GMM sebesar {abs(delta):.2f}% WER")
    else:
        print(f"  = TDNN dan GMM memiliki WER yang sama")


# ─────────────────────────────────────────────────────────────
#  MAIN PIPELINE
# ─────────────────────────────────────────────────────────────

def main():
    np.random.seed(RANDOM_SEED)

    print("=" * 60)
    print("  TAHAP 1 — Memuat Data & Ekstraksi MFCC dari WAV")
    print("=" * 60)
    print(f"  Folder audio  : {SAMPLED_ROOT}")
    print(f"  CSV transkrip : {CSV_PATH}")
    print(f"  Output TDNN   : {DIR_SAVE}")
    print(f"  Input dim     : {INPUT_DIM} (MFCC {N_MFCC} + delta + delta2)")

    sequences, labels, label2id, id2label, word_refs, actual_dim = \
        load_data_from_wav(CSV_PATH)

    all_ids = list(sequences.keys())
    train_ids, val_ids, test_ids = split_dataset(all_ids)

    print("\n" + "=" * 60)
    print("  TAHAP 2 — Bangun & Latih Model TDNN")
    print("=" * 60)
    print(f"  Arsitektur TDNN:")
    print(f"    Input dim  : {actual_dim}")
    for i, (h, ctx) in enumerate(zip(HIDDEN_DIMS, TDNN_CONTEXTS)):
        print(f"    Layer {i+1}     : {h} unit  context {ctx}")
    print(f"    Output dim : {len(label2id)} kelas")
    print(f"    Dropout    : {DROPOUT_RATE}")
    print(f"    Optimizer  : Adam  lr={LEARNING_RATE}")
    print(f"    Batch size : {BATCH_SIZE}")

    model = TDNNModel(
        input_dim   = actual_dim,
        hidden_dims = HIDDEN_DIMS,
        contexts    = TDNN_CONTEXTS,
        output_dim  = len(label2id),
        dropout     = DROPOUT_RATE,
        lr          = LEARNING_RATE,
    )

    history = train(model, train_ids, val_ids, sequences, labels)
    pd.DataFrame(history).to_csv(OUT_HISTORY_CSV, index=False)
    print(f"  {OUT_HISTORY_CSV} tersimpan")

    if os.path.exists(MODEL_PATH):
        model.load(MODEL_PATH)

    print("\n" + "=" * 60)
    print("  TAHAP 3 — Evaluasi WER + Akurasi (Val + Test)")
    print("=" * 60)

    eval_ids = val_ids + test_ids
    wer_rows, acc_rows, summary = evaluate(
        model, eval_ids, sequences, labels, id2label, word_refs
    )

    df_wer = pd.DataFrame(wer_rows)
    df_acc = pd.DataFrame(acc_rows)
    df_dec = df_wer[["audio_id", "referensi", "hipotesis", "WER"]]

    df_wer[["audio_id","WER","S","D","I","N"]].to_csv(OUT_WER_CSV, index=False)
    df_dec.to_csv(OUT_DEC_CSV, index=False)
    df_acc.to_csv(OUT_ACC_CSV, index=False)

    print(f"\n  {OUT_WER_CSV}")
    print(f"  {OUT_DEC_CSV}")
    print(f"  {OUT_ACC_CSV}")

    print("\n" + "=" * 60)
    print("  TAHAP 4 — Perbandingan GMM vs TDNN")
    print("=" * 60)
    print_comparison(summary)

    return model, summary, history


# ─────────────────────────────────────────────────────────────

if __name__ == "__main__":
    model, summary, history = main()
PYEOF